# Shamba Rafiki — Leaf Disease Classifier (Google Colab, GPU)

Trains **MobileNetV3-small** on **maize** (PlantVillage) + **beans** (iBean) and exports two files for the backend:

- `plant_classifier.onnx` — the CPU inference model
- `plant_classifier.labels.json` — the class → (crop, condition) map

You drop both into your repo's `models/` folder and the app auto-detects them. Training runs on Colab's free GPU (a few minutes); your own machine only ever *runs* the exported model via onnxruntime (milliseconds), so nothing heats up your laptop.

**Before you run:** menu → **Runtime → Change runtime type → Hardware accelerator = GPU → Save.** Then run the cells top to bottom.

> The preprocessing here (224×224, ImageNet normalization, resize-shorter-side + center-crop for eval) is kept identical to `backend/app/vision/preprocess.py`, so training and serving never drift.

## 1. Confirm the GPU is on

In [ ]:
import torch
print('Torch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('No GPU! Runtime > Change runtime type > GPU, then re-run this cell.')

## 2. Install the extra libraries

Colab already has torch + torchvision; we only add `datasets` (for beans) and `onnx`.

In [ ]:
!pip -q install datasets onnx onnxruntime onnxscript


## 3. Download the data and assemble an ImageFolder

- **Maize**: the four `Corn_(maize)___*` classes from the PlantVillage dataset (GitHub, `spMohanty`).
- **Beans**: the iBean/Makerere dataset from Hugging Face (`angular_leaf_spot`, `bean_rust`, `healthy`).

Folders are named `Crop___Condition` so the label map is derived automatically. `CAP_PER_CLASS` bounds the images per class to keep it quick — raise it (or set to `None`) for a stronger model.

In [ ]:
import os, shutil, random, subprocess
from pathlib import Path

random.seed(42)
OUT = Path('data/plant_images/train')
OUT.mkdir(parents=True, exist_ok=True)
CAP_PER_CLASS = 1000  # set to None for all images (slower, stronger)

# --- Maize: clone PlantVillage (shallow) and copy the Corn classes ---
if not Path('PlantVillage-Dataset').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/spMohanty/PlantVillage-Dataset.git'], check=True)

color = Path('PlantVillage-Dataset/raw/color')
maize_dirs = sorted(d for d in color.iterdir() if d.name.startswith('Corn_(maize)___'))
for d in maize_dirs:
    imgs = list(d.glob('*.jpg')) + list(d.glob('*.JPG')) + list(d.glob('*.png'))
    random.shuffle(imgs)
    if CAP_PER_CLASS:
        imgs = imgs[:CAP_PER_CLASS]
    dest = OUT / d.name
    dest.mkdir(parents=True, exist_ok=True)
    for p in imgs:
        shutil.copy(p, dest / p.name)
    print(f'{d.name:45s} {len(imgs)} images')

# --- Beans: Hugging Face iBean dataset ---
from datasets import load_dataset
beans = load_dataset('AI-Lab-Makerere/beans')  # iBean dataset (namespaced id)
print('Beans label order:', beans['train'].features['labels'].names)
bean_names = {i: n for i, n in enumerate(beans['train'].features['labels'].names)}
pretty = {'angular_leaf_spot': 'Angular_leaf_spot', 'bean_rust': 'Bean_rust', 'healthy': 'healthy'}

for split in ['train', 'validation', 'test']:
    for i, ex in enumerate(beans[split]):
        cond = pretty.get(bean_names[ex['labels']], bean_names[ex['labels']])
        dest = OUT / f'Beans___{cond}'
        dest.mkdir(parents=True, exist_ok=True)
        ex['image'].convert('RGB').save(dest / f'{split}_{i}.jpg')

print('\nClasses assembled:')
for c in sorted(os.listdir(OUT)):
    print(' ', c, '=', len(os.listdir(OUT / c)))

## 4. Train MobileNetV3-small (transfer learning)

ImageNet-pretrained backbone, fresh classification head sized to our classes. ~8 epochs is plenty on this data; watch `val_acc` climb.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V3_Small_Weights

SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
DATA = 'data/plant_images/train'
EPOCHS = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training on', device)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(SIZE),
    transforms.CenterCrop(SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full = datasets.ImageFolder(DATA, transform=train_tf)
classes = full.classes
n_val = int(len(full) * 0.15)
n_tr = len(full) - n_val
g = torch.Generator().manual_seed(42)
tr_split, va_split = random_split(full, [n_tr, n_val], generator=g)
val_base = datasets.ImageFolder(DATA, transform=eval_tf)  # eval transform, no aug
va_ds = Subset(val_base, va_split.indices)

tl = DataLoader(tr_split, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
vl = DataLoader(va_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f'Classes ({len(classes)}): {classes}')
print(f'Train {len(tr_split)}  Val {len(va_ds)}')

model = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_acc, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for x, y in tl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in vl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.numel()
    acc = correct / max(1, total)
    print(f'epoch {epoch:>2}/{EPOCHS}  loss={running/len(tr_split):.4f}  val_acc={acc:.3f}')
    if acc >= best_acc:
        best_acc = acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print('Best val accuracy:', round(best_acc, 3))

## 5. Export `plant_classifier.onnx` + `plant_classifier.labels.json`

The label map is derived from the folder names using the same rules as `training/train_classifier.py`, and written in the exact shape the backend's `ClassLabels` loader expects.

In [ ]:
import json
from pathlib import Path

CROP_CANON = {'corn': 'maize', 'corn_(maize)': 'maize', 'maize': 'maize',
              'beans': 'beans', 'bean': 'beans', 'tomato': 'tomato', 'cassava': 'cassava'}

def canon_crop(tok):
    t = tok.strip().lower().replace(' ', '_')
    if t in CROP_CANON:
        return CROP_CANON[t]
    return CROP_CANON.get(t.split('_')[0].split('(')[0], 'unknown')

def clean_condition(tok):
    t = tok.replace('_', ' ').strip()
    if not t:
        return 'unknown'
    if t.lower() in ('healthy', 'health'):
        return 'healthy'
    return ' '.join(w if (w.isupper() and len(w) <= 4) else w.capitalize() for w in t.split())

def parse_class(name):
    crop_tok, cond_tok = (name.split('___', 1) + [''])[:2] if '___' in name else ('', name)
    return {'crop': canon_crop(crop_tok), 'condition': clean_condition(cond_tok), 'label': name}

labels = [parse_class(c) for c in classes]
Path('plant_classifier.labels.json').write_text(json.dumps(labels, indent=2))
print(json.dumps(labels, indent=2))

model.eval().cpu()
dummy = torch.randn(1, 3, SIZE, SIZE)
torch.onnx.export(
    model, dummy, 'plant_classifier.onnx',
    input_names=['input'], output_names=['logits'], opset_version=13,
)
print('\nExported plant_classifier.onnx')
# Embed weights so the ONNX is ONE self-contained file (no .onnx.data sidecar)
import onnx as _onnx
_onnx.save_model(_onnx.load('plant_classifier.onnx'), 'plant_classifier.onnx', save_as_external_data=False)
print('Embedded weights -> single-file plant_classifier.onnx')


## 6. Sanity-check the exported model with onnxruntime

Confirms the ONNX loads and predicts a real class on a held-out image — the same runtime your backend uses.

In [ ]:
import numpy as np, onnxruntime as ort
from PIL import Image

def preprocess(path):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    s = SIZE / min(w, h)
    img = img.resize((max(SIZE, round(w*s)), max(SIZE, round(h*s))))
    W, H = img.size
    l, t = (W - SIZE)//2, (H - SIZE)//2
    img = img.crop((l, t, l+SIZE, t+SIZE))
    a = np.asarray(img, np.float32)/255.0
    a = (a - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return np.transpose(a, (2,0,1))[None].astype(np.float32)

sess = ort.InferenceSession('plant_classifier.onnx', providers=['CPUExecutionProvider'])
sample = next((OUT).rglob('*.jpg'))
logits = sess.run(None, {'input': preprocess(sample)})[0][0]
probs = np.exp(logits - logits.max()); probs /= probs.sum()
top = probs.argsort()[::-1][:3]
print('Sample:', sample)
for i in top:
    print(f'  {classes[i]:45s} {probs[i]:.3f}')

## 7. Download the two files

Then drop both into your repo's **`models/`** folder:

```
RafikiAI/models/plant_classifier.onnx
RafikiAI/models/plant_classifier.labels.json
```

Restart the backend. `/health` will show `"classifier": {"available": true}` and `/classify` (and the kiosk UI's photo upload) will start returning real diagnoses.

In [ ]:
from google.colab import files
files.download('plant_classifier.onnx')
files.download('plant_classifier.labels.json')